In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pystac-client",
    "planetary-computer",
    "stackstac",
    "rioxarray",
    "geopandas",
    "shapely",
    "xarray",
    "numpy",
    "pandas",
    "-q",
], check=True)

In [ ]:
import os
print("=== Kaggle input layout ===")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level <= 3:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 3:
            for f in files[:5]:
                print(f'  {indent}{f}')

In [ ]:
import sys, os


def find_farmtrust_core():
    search_roots = ['/kaggle/input', '/kaggle/input/datasets']
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if os.path.basename(dirpath) == 'farmtrust_core':
                parent = os.path.dirname(dirpath)
                print(f"Found farmtrust_core at: {dirpath}")
                print(f"Adding to sys.path: {parent}")
                return parent
    raise RuntimeError(
        f"farmtrust_core not found under /kaggle/input. "
        f"Available: {os.listdir('/kaggle/input')}"
    )


sys.path.insert(0, find_farmtrust_core())
import farmtrust_core
print("farmtrust_core imported successfully")

In [ ]:
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'
MAX_CLOUD = 30


def bbox_to_polygon(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    return {
        'type': 'Polygon',
        'coordinates': [
            [
                [min_lon, min_lat],
                [max_lon, min_lat],
                [max_lon, max_lat],
                [min_lon, max_lat],
                [min_lon, min_lat],
            ]
        ],
    }


AOIS = [
    {
        'aoi_id': 'aoi_demo_02',
        'governorate': 'Demo-region offset east',
        'bbox': [31.0403, 30.594356, 31.063848, 30.616635],
    },
    {
        'aoi_id': 'aoi_demo_03',
        'governorate': 'Demo-region offset west',
        'bbox': [30.9403, 30.594356, 30.963848, 30.616635],
    },
    {
        'aoi_id': 'aoi_demo_04',
        'governorate': 'Demo-region offset north',
        'bbox': [30.9903, 30.644356, 31.013848, 30.666635],
    },
    {
        'aoi_id': 'aoi_demo_05',
        'governorate': 'Demo-region offset south',
        'bbox': [30.9903, 30.544356, 31.013848, 30.566635],
    },
    {
        'aoi_id': 'aoi_demo_06',
        'governorate': 'Demo-region offset northeast',
        'bbox': [31.0903, 30.694356, 31.113848, 30.716635],
    },
    {
        'aoi_id': 'aoi_demo_07',
        'governorate': 'Demo-region offset northwest',
        'bbox': [30.8903, 30.694356, 30.913848, 30.716635],
    },
    {
        'aoi_id': 'aoi_demo_08',
        'governorate': 'Demo-region offset southeast',
        'bbox': [31.0903, 30.494356, 31.113848, 30.516635],
    },
    {
        'aoi_id': 'aoi_demo_09',
        'governorate': 'Demo-region offset southwest',
        'bbox': [30.8903, 30.494356, 30.913848, 30.516635],
    },
    {
        'aoi_id': 'aoi_demo_10',
        'governorate': 'Demo-region offset far east',
        'bbox': [31.1903, 30.644356, 31.213848, 30.666635],
    },
    {
        'aoi_id': 'aoi_demo_11',
        'governorate': 'Demo-region offset far west',
        'bbox': [30.7903, 30.544356, 30.813848, 30.566635],
    },
]

for aoi in AOIS:
    aoi['geometry'] = bbox_to_polygon(aoi['bbox'])

print(f'Defined {len(AOIS)} AOIs from {START_DATE} to {END_DATE}, max_cloud={MAX_CLOUD}')

In [ ]:
from pathlib import Path
import importlib
import traceback


ingest_mod = importlib.import_module('farmtrust_core.ingest.pipeline')
print("Available in ingest.pipeline:", dir(ingest_mod))

# Find the main ingest function dynamically.
ingest_fn = None
for name in ['run_ingest', 'ingest', 'run', 'main', 'ingest_aoi', 'write_outputs']:
    if hasattr(ingest_mod, name):
        ingest_fn = getattr(ingest_mod, name)
        print(f"Using ingest function: {name}")
        break

if ingest_fn is None:
    raise RuntimeError(f"No ingest function found. Available: {dir(ingest_mod)}")


WORKING_ROOT = Path('/kaggle/working')
succeeded = []
failed = []

for index, aoi in enumerate(AOIS, start=1):
    aoi_id = aoi['aoi_id']
    output_dir = WORKING_ROOT / aoi_id
    print(f'[{index}/{len(AOIS)}] INGEST {aoi_id} - {aoi["governorate"]}', flush=True)
    try:
        ingest_fn(
            output_dir=output_dir,
            aoi_id=aoi_id,
            bbox=aoi['bbox'],
            start_date=START_DATE,
            end_date=END_DATE,
            max_cloud=float(MAX_CLOUD),
            force_rerun=False,
            limit_items=None,
            log_signed_hrefs=False,
            deduplicate=True,
            max_workers=4,
            geometry=aoi['geometry'],
        )
        csv_path = output_dir / 'indices_timeseries.csv'
        if not csv_path.exists():
            raise FileNotFoundError(f'Missing expected output: {csv_path}')
        print(f'SUCCESS {aoi_id} {csv_path}', flush=True)
        succeeded.append(aoi_id)
    except Exception as exc:
        print(f'FAILED {aoi_id}: {type(exc).__name__}: {exc}', flush=True)
        traceback.print_exc()
        failed.append(aoi_id)

print('Ingest summary')
print(f'  succeeded: {len(succeeded)}')
print(f'  failed: {len(failed)}')
print(f'  succeeded AOIs: {succeeded}')
print(f'  failed AOIs: {failed}')

In [ ]:
import os
import subprocess
from pathlib import Path

aoi_dirs = sorted(str(p) for p in Path("/kaggle/working").glob("aoi_*"))
if not aoi_dirs:
    raise RuntimeError("No AOI output directories found under /kaggle/working")

subprocess.run([
    "zip",
    "-r",
    "/kaggle/working/egypt_aoi_timeseries.zip",
    *aoi_dirs,
], check=True)

size = os.path.getsize("/kaggle/working/egypt_aoi_timeseries.zip") / 1e6
print(f"Output zip: {size:.1f} MB")

In [ ]:
print('=== INGEST COMPLETE. Download: egypt_aoi_timeseries.zip ===')
